# Compress repaired MusicFM FP32, keep the rest of the pipeline fixed
Gunakan kernel SongFormer Python3.11 di Mac, root Section-Annotations.
Setiap kandidat berasal dari MusicFM FP32 yang sudah lolos exact-window dan full
validation. Tidak mengompresi model MusicFM lama yang menghasilkan NaN.
Frontend, MuQ, rotary dan head FP32 tetap sama. Fitur MuQ dari baseline dipakai
ulang setelah pemeriksaan hash/runtime; hanya MusicFM dihitung ulang. Kandidat
memakai cache terpisah dengan ruang disk tambahan. Kompresi ini bobot saja,
bukan kuantisasi aktivasi atau janji komputasi integer/speedup iPhone.


In [ ]:
import sys, subprocess, json
from pathlib import Path
from datetime import datetime
from finetuning.common import sha_file
assert Path('finetuning/config.json').is_file()
SONGMAP_ROOT=Path('/Users/danendra/Documents/ADA26/C6/SongMap')
paths=sorted(Path('outputs.local').glob('musicfm-fp32-*/all-validation.json'),
    key=lambda p:p.stat().st_mtime,reverse=True)
valid=[]
for path in paths:
    report=json.loads(path.read_text())
    repair=path.parent/'repair_report.json'
    override=report.get('musicfm_override') or {}
    if (report.get('full_validation') and repair.is_file() and not override.get('compression_scheme')
            and override.get('report_sha256')==sha_file(repair)):
        valid.append(path)
assert valid,'Full-validation MusicFM FP32 yang cocok belum ditemukan'
baseline_path=valid[0]
baseline=json.loads(baseline_path.read_text())
repair_path=baseline_path.parent/'repair_report.json'
reuse_cache=Path(baseline['diagnostic_cache'])
assert reuse_cache.is_dir(),'Cache baseline MuQ harus tersedia'
head_proofs=sorted(Path('outputs.local').glob('coreml-comparison-*/fp32-validation.json'),
    key=lambda p:p.stat().st_mtime,reverse=True)
head_proofs=[p for p in head_proofs if (lambda r:r.get('full_validation') and r.get('logit_parity_passed')
    and Path(r['package']).resolve()==Path(baseline['fp32_head']).resolve())(json.loads(p.read_text()))]
assert head_proofs,'Bukti validasi head FP32 baseline belum ditemukan'
head_proof=head_proofs[0]
experiment=Path('outputs.local')/('musicfm-compression-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
experiment.mkdir(exist_ok=False)
print('Baseline pipeline:',baseline_path)
print('Baseline F1:',baseline['coreml_encoders_with_fp32_head']['macro_f1_present'])
print('Reusing unchanged MuQ:',reuse_cache)
print('Output:',experiment)

def run_logged(command,log_path):
    with log_path.open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
            code=process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if code:
        raise RuntimeError(f'Proses gagal ({code}); lihat {log_path}')


Buat kandidat INT8 dan palette6 secara independen. Masing-masing diuji tiga kali pada window Pilihanku yang sebelumnya menghasilkan NaN. Kmeans palette memakai satu worker dan bisa memerlukan waktu. Jika NaN muncul, berhenti: kandidat tidak boleh dilanjutkan.

In [ ]:
for scheme in ('int8','palette6'):
    saved=experiment/scheme/'compression_report.json'
    if saved.is_file():
        prior=json.loads(saved.read_text())
        assert prior.get('window_finite_passed') and prior['compression_scheme']==scheme
        assert prior['source_repair_report_sha256']==sha_file(repair_path)
        assert prior['source_pipeline_report_sha256']==sha_file(baseline_path)
        assert Path(prior['package']).is_dir()
        print('Reuse completed compression:',scheme)
        continue
    run_logged([sys.executable,'-u','-m','deployment.compress_musicfm',
        '--repair-report',str(repair_path),'--pipeline-report',str(baseline_path),
        '--scheme',scheme,'--output-dir',str(experiment/scheme)],
        experiment/f'{scheme}-compression.log')


Evaluasi satu lagu terlebih dahulu, dengan head FP32 yang sama. Fitur MuQ baseline dipakai ulang, tidak dijalankan ulang.

In [ ]:
def evaluate_candidate(scheme,limit,name):
    existing=experiment/f'{scheme}-{name}.json'
    if existing.is_file():
        prior=json.loads(existing.read_text())
        assert prior['musicfm_override']['report_sha256']==sha_file(experiment/scheme/'compression_report.json')
        assert prior['plan_id']==baseline['plan_id']
        if limit==0:
            assert prior['annotations_id']==baseline['annotations_id'] and prior['full_validation']
        else:
            assert prior['tracks']==1
        print('Reuse completed evaluation:',existing)
        return
    run_logged([sys.executable,'-u','-m','deployment.evaluate_pipeline',
        '--songmap-root',str(SONGMAP_ROOT),'--fp32-validation-report',str(head_proof),
        '--musicfm-export-report',str(experiment/scheme/'compression_report.json'),
        '--reuse-muq-cache',str(reuse_cache),
        '--cache-dir',str(experiment/scheme/'validation-cache'),
        '--limit',str(limit),'--output',str(experiment/f'{scheme}-{name}.json')],
        experiment/f'{scheme}-{name}.log')

for scheme in ('int8','palette6'):
    evaluate_candidate(scheme,1,'one-track')


Setelah smoke selesai tanpa runtime error, evaluasi seluruh validation. Tidak memakai test; catat perubahan kelas individual, bukan hanya nilai F1 gabungan. Cache kandidat yang sudah selesai dilewati jika run dilanjutkan.

In [ ]:
for scheme in ('int8','palette6'):
    evaluate_candidate(scheme,0,'all-validation')


In [ ]:
baseline_metrics=baseline['coreml_encoders_with_fp32_head']
summary={'baseline_pipeline_report':str(baseline_path),'baseline_macro_f1':baseline_metrics['macro_f1_present'],'candidates':[]}
for scheme in ('int8','palette6'):
    compression=json.loads((experiment/scheme/'compression_report.json').read_text())
    report=json.loads((experiment/f'{scheme}-all-validation.json').read_text())
    assert report['full_validation']
    for key in ('plan_id','provenance_id','annotations_id','checkpoint_sha256','tracks','fp32_head'):
        assert report[key]==baseline[key],f'Perbandingan berbeda: {key}'
    metrics=report['coreml_encoders_with_fp32_head']
    item={'scheme':scheme,'musicfm_size_mib':compression['candidate_bytes']/1024**2,
        'musicfm_size_reduction_fraction':compression['size_reduction_fraction'],
        'macro_f1':metrics['macro_f1_present'],
        'f1_delta_from_repaired_pipeline':metrics['macro_f1_present']-baseline_metrics['macro_f1_present'],
        'section_accuracy':metrics['section_accuracy'],
        'low_coverage_sections':metrics['low_coverage_sections'],
        'section_agreement_with_training_cache':report['section_prediction_agreement'],
        'ready_for_app_integration':False}
    print(json.dumps(item,indent=2))
    print('Per-class F1:',metrics['per_class'])
    summary['candidates'].append(item)
(experiment/'comparison.json').write_text(json.dumps(summary,indent=2))
print('Share:',experiment/'comparison.json')
print('Kecepatan iPhone belum diukur untuk encoder kandidat.')
